In [ ]:
from pathlib import Path
import pandas as pd


# Caminho relativo à raiz: altere para selecionar outro CSV local.
caminho_csv = Path("dados tratados/base_modelo.csv")  # Forneça este arquivo localmente ou ajuste o caminho.
separador = ";"
decimal = "."
arquivo_csv = raiz_projeto / caminho_csv
if not arquivo_csv.is_file():
    raise FileNotFoundError(f"CSV não encontrado: {arquivo_csv}. Ajuste caminho_csv.")

dados = pd.read_csv(arquivo_csv, sep=separador, decimal=decimal, encoding="utf-8-sig")

print(f"Arquivo carregado: {caminho_csv}")
print(f"Dimensões: {dados.shape[0]} linhas e {dados.shape[1]} colunas")
# A base CSV deve ser fornecida localmente.
# As demais análises exigem a base preparada descrita no README.


In [ ]:
# Conferir as primeiras linhas
display(dados.head())

# Conferir nomes e tipos das colunas e valores preenchidos
dados.info()

# Contar valores ausentes por coluna
print("\nValores ausentes por coluna:")
display(dados.isna().sum())

# Contar linhas duplicadas
print(f"\nLinhas duplicadas: {dados.duplicated().sum()}")

# Resumo das colunas numéricas
display(dados.describe())

In [ ]:
area_por_talhao = dados[["talhao", "area_ha"]].drop_duplicates()

print(f"Quantidade de talhões: {area_por_talhao['talhao'].nunique()}")
print(f"Área total cadastrada: {area_por_talhao['area_ha'].sum()} hectares")

In [ ]:
# Coluna que contém o valor que queremos prever
coluna_alvo = "atr_kg_ton"

# Lista inicial das características que o modelo usará como entradas.
# Incluímos solo, soqueira, variedade, corte, chuva, daninhas,
# herbicida e safra para podermos avaliar essas variáveis.
colunas_entrada = [
    "falhas_soqueira_pct",
    "vigor_soqueira_pct",
    "variedade",
    "textura_solo",
    "numero_corte",
    "safra"
]

# X guarda as entradas do modelo.
# O .copy() cria uma cópia independente dessas colunas.
X = dados[colunas_entrada].copy()

# y guarda os valores de ATR correspondentes a cada linha de X.
y = dados[coluna_alvo].copy()

# Separamos as colunas categóricas (texto) das numéricas.
# Modelos de machine learning não costumam usar texto diretamente;
# trataremos essas categorias numa etapa posterior.
colunas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
colunas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

# Conferências para garantir que a seleção foi feita corretamente.
assert len(X) == len(y), "X e y precisam ter o mesmo número de linhas."
assert X.index.equals(y.index), "As entradas e os alvos precisam manter a correspondência entre linhas."
assert X.isna().sum().sum() == 0, "Há valores ausentes nas entradas."
assert y.isna().sum() == 0, "Há valores ausentes no alvo."

# Mostramos um resumo da separação.
print(f"Quantidade de registros: {len(X)}")
print(f"Quantidade de variáveis de entrada: {X.shape[1]}")
print(f"Variável alvo: {coluna_alvo}")
print(f"Colunas categóricas: {colunas_categoricas}")
print(f"Quantidade de colunas numéricas: {len(colunas_numericas)}")

# Visualizamos as primeiras entradas e os respectivos valores de ATR.
display(X.head().assign(atr_kg_ton=y.head()))

In [ ]:
# Construindo modelo de treinamento
# - mascara_treino = dados["safra"].isin(anos_treino) - replicando linha de código com for e if/else

anos_treino = list(range(2016, 2024))  # 2016 até 2023
anos_teste = [2024, 2025]
mascara_treino = []
mascara_teste = []

# Coluna que contém o valor que queremos prever
coluna_alvo = "atr_kg_ton"

# Lista inicial das características que o modelo usará como entradas.
# Incluímos solo, soqueira, variedade, corte, chuva, daninhas,
# herbicida e safra para podermos avaliar essas variáveis.
colunas_entrada = [
    "falhas_soqueira_pct",
    "vigor_soqueira_pct",
    "variedade",
    "textura_solo",
    "numero_corte",
    "safra"
]

# X guarda as entradas do modelo.
# O .copy() cria uma cópia independente dessas colunas.
X = dados[colunas_entrada].copy()

# y guarda os valores de ATR correspondentes a cada linha de X.
y = dados[coluna_alvo].copy()

for safra in dados['safra']:
    if safra in anos_treino:
        mascara_treino.append(True)
        mascara_teste.append(False)

    else:
        mascara_treino.append(False)
        mascara_teste.append(True)


#print(mascara_treino)
#print(mascara_teste)

X_treino = X.loc[mascara_treino]
#print(X_treino)
X_teste = X.loc[mascara_teste]
print(X_teste)

y_treino = y.loc[mascara_treino]

y_teste = y.loc[mascara_teste]
print(y_teste)



In [ ]:
# Criamos cópias para preservar os DataFrames originais.
X_treino_numerico = X_treino.copy()
X_teste_numerico = X_teste.copy()

# Definimos códigos para cada variedade.
# Os códigos são identificadores, não uma ordem de qualidade.
codigos_variedade = {
    "RB867515": 1,
    "RB966928": 2,
    "CTC4": 3
}

# Definimos códigos para cada textura de solo.
codigos_textura = {
    "argiloso": 1,
    "medio": 2,
    "arenoso": 3
}

# Percorremos as linhas do treino e substituímos os textos pelos códigos.
for indice in X_treino_numerico.index:
    variedade = X_treino_numerico.loc[indice, "variedade"]
    textura = X_treino_numerico.loc[indice, "textura_solo"]

    X_treino_numerico.loc[indice, "variedade"] = codigos_variedade[variedade]
    X_treino_numerico.loc[indice, "textura_solo"] = codigos_textura[textura]

# Fazemos a mesma conversão no teste, usando os mesmos códigos.
for indice in X_teste_numerico.index:
    variedade = X_teste_numerico.loc[indice, "variedade"]
    textura = X_teste_numerico.loc[indice, "textura_solo"]

    X_teste_numerico.loc[indice, "variedade"] = codigos_variedade[variedade]
    X_teste_numerico.loc[indice, "textura_solo"] = codigos_textura[textura]

# Conferimos se as colunas de texto foram convertidas.
print(X_treino_numerico.head())
print(X_treino_numerico.dtypes)

In [ ]:
# Convertemos as duas colunas categóricas para números inteiros.
X_treino_numerico["variedade"] = X_treino_numerico["variedade"].astype(int)
X_treino_numerico["textura_solo"] = X_treino_numerico["textura_solo"].astype(int)

X_teste_numerico["variedade"] = X_teste_numerico["variedade"].astype(int)
X_teste_numerico["textura_solo"] = X_teste_numerico["textura_solo"].astype(int)

# Conferimos se agora todas as entradas são numéricas.
print(X_treino_numerico.dtypes)

In [ ]:
# Transformamos as colunas em listas de valores.
x = X_treino_numerico["vigor_soqueira_pct"].tolist()
valores_atr = y_treino.tolist()

# Calculamos a média de cada lista.
media_x = sum(x) / len(x)
media_y = sum(valores_atr) / len(valores_atr)

print("Média do vigor:", media_x)
print("Média do ATR:", media_y)